# Lab 1A - Introduction to the RPi, Linux, and ROS

**Cours:** GPA778 - Mobile Robotic Systems

---

## Table of Contents

1. [Introduction](#introduction)
2. [Learning Objectives](#objectifs)
3. [Milestone 1: RPi Hardware and Software Features](#jalon1)
4. [Milestone 2: Starting Up and First Interactions with the RPi](#jalon2)
   - [Robot <-> Computer Connection](#connexion)
   - [Graphical Remote Access via VNC](#vnc)
   - [Connection via SSH](#ssh)
   - [File Transfer via SCP](#scp)
5. [Milestone 3: Navigating Linux and Using the Terminal](#jalon3)
   - [Exercise #1: File-Creation Script](#exercice1)
6. [Milestone 4: First Contact with ROS](#jalon4)
   - [Exercise #2: The bavardage Package](#exercice2)
7. [Additional Questions](#questions)
8. [Final Demonstration](#demo)

<a id='introduction'></a>
## Introduction

The goal of this first lab is to familiarize you with the various hardware and software components we will use throughout the term. We will first introduce the hardware of the **Raspberry Pi 4** (RPi 4) in detail. Next, we will explore the **Linux** operating system (Ubuntu 22.04 LTS - *Jammy Jellyfish*) through a series of exercises. Finally, you will create your first **ROS package** in Python.

This package will exchange "Hello World!" messages between a first *node*, acting as a **publisher**, and a second *node*, acting as a **subscriber**.

<a id='objectifs'></a>
## Learning Objectives

The main objective of this lab is to introduce you to the onboard computer of the mobile robot we will use throughout the term and to build your knowledge of the RPi's hardware and software.

### Hardware
Develop a detailed understanding of input/output peripherals:
- GPIOs
- USB, HDMI, audio, and camera ports
- System specifications (memory, CPU, communication buses/architecture)

### Software
Gradually become familiar with:
- **Linux** (Ubuntu 22.04)
- **ROS** (ROS 2)
- Notions fondamentales de **Python**

### Specific Objectives

By the end of this lab, you will be able to:

- ✓ Describe the RPi hardware (CPU, memory, communication buses, and input/output peripherals)
- ✓ Use Ubuntu *Jammy Jellyfish* and navigate the file system through the graphical interface
- ✓ Use the command-line terminal and BASH interpreter to:
  - Create, modify, and delete files
  - Create directories
  - Change permissions (read, write, execute)
- ✓ Run common BASH commands and create simple scripts
- ✓ Create and initialize a ROS workspace
- ✓ Create a first ROS package named "bavardage" to establish inter-process communication
- ✓ Program two nodes (*talker* and *listener*): the first will publish data and the second will receive it

<a id='jalon1'></a>
## Milestone 1: RPi Hardware and Software Features

### Raspberry Pi 4 Model B

The Raspberry Pi 4 Model B is a **single-board computer**. Like any computer system, the RPi has four essential components:

1. **Processor**
2. **Memory**
3. **Input/output peripherals**
4. **Communication buses**

### Main Components
![image.png](https://www.etechnophiles.com/wp-content/uploads/2021/01/R-Pi-4-GPIO-Pinout-1.jpg)
#### Connectors and Peripherals

Starting from the upper-left corner (counterclockwise):

- **USB-C**: Power supply (15W minimum: 5.1V @ 3A)
- **2× micro-HDMI**: 4K video output to two displays simultaneously
- **MIPI-CSI**: Camera connector (Mobile Industry Processor Interface - Serial Camera Interface)
- **Audio connector**: 3.5 mm audio output
- **USB 2.0 and 3.0 ports**: Multiple peripheral connectors
- **Ethernet**: Wired network connection
- **Header 40 pins**: Access to GPIOs (General Purpose Inputs/Outputs)
- **DPI**: Display Parallel Interface (24 bits)

### The Broadcom BCM2711 Processor

Key specifications:
- **Architecture**: Quad-core Cortex-A72 (ARM v8) at 1.5 GHz
- **4 cores**: Enables parallel scheduling
- **64-bit**: Improves memory-access efficiency
- **ALU**: 4 arithmetic logic units

This multicore architecture justifies using a full operating system (Ubuntu) to manage scheduling and access to peripherals and memory.

### Memory

#### Random Access Memory (RAM)
- **Capacity**: 8 GB (maximum supported)
- **Type**: Volatile (lost when powered off)
- **Use**: Loading data for applications (e.g., point clouds and algorithms)
- **Advantage**: Very short access time

#### Non-volatile Memory (SD Card)
- **Capacity**: 64 GB
- **Type**: Persistent flash memory
- **Contents**:
  - Ubuntu operating system
  - User files
  - Algorithms and code
- **Advantage**: Data retained after shutdown

### Communication Buses

Bus types:
- **Data bus**: Data transfer
- **Address bus**: Memory addressing
- **Control bus**: Control signals
- **Sync bus**: Synchronization

The RPi 4 supports a **64-bit** architecture, allowing 64 bits of data to be transferred per clock cycle between the processor and memory.

### GPIOs (General Purpose Inputs/Outputs)

The 40-pin header provides access to the GPIOs. Pin functions:

- **Fixed pins**:
  - Power supply 3.3V
  - Power supply 5V
  - Ground (GND)

- **Configurable pins** (GPIO #2 through #27):
  - Digital inputs/outputs
  - PWM (Pulse-Width Modulation)
  - Communication I2C
  - Communication SPI
  - UART

**Note**: GPIOs #0 and #1 are not available on the header.

#### GPIO Alternate Functions

Each GPIO can be configured for different functions (see the table in the PDF):
- **ALT0-ALT5**: Different operating modes
- Examples: SDA/SCL (I2C), MOSI/MISO (SPI), TXD/RXD (UART), PWM, etc.

### Applications for the Mobile Robot

In this project, the GPIOs are used for:
- **Digital inputs**: Read wheel-encoder pulses
- **Digital outputs**: Turn on LEDs on the board
- **Mode PWM**: Adjusting motor speed and LIDAR tilt
- **Communication I2C**: Communicating with the inertial measurement unit

<a id='jalon2'></a>
## Milestone 2: Starting Up and First Interactions with the RPi

### 1. Login Information

Before starting, note these essential details:

- **Username**: `ubuntu`
- **sudo password**: `raspberry`

**Important**: Make sure the RPi is powered on (red LED lit) and wait at least **1 minute** after powering it on to allow the operating system to boot.

### 2. Ways to Interact with the RPi

Three main methods are available:

1. **SSH** (*Secure Shell*): Remote command-line terminal
2. **SCP** (*Secure Copy*): File transfer
3. **Keyboard**: Use the terminal on the Raspberry Pi through an HDMI-connected display and keyboard. The cable must be connected before starting the RPi

Choose the method best suited to your task.

<a id='connexion'></a>
### 3. Robot <-> Computer Connection and Obtaining an IP Address

To exchange data between the RPi and a computer, we will use the **TCP/IP** protocol. Two methods are available:


#### 3.1 Connect the robot to the network:
To communicate with the robot remotely, it must be connected to the ETS-IdO network. Each robot has an IP address reserved through DHCP.
1. Use the HDMI cable, lab monitor, and keyboard to access the RPi. Connect the HDMI cable before turning on the Raspberry Pi.
2. Ubuntu 22.04 uses [Netplan](https://documentation.ubuntu.com/server/explanation/networking/configuring-networks/) to manage Wi-Fi and Ethernet connections. Open the file `/etc/netplan/50-cloud-init.yaml` with the nano text editor:
```bash
sudo nano /etc/netplan/50-cloud-init.yaml
```

3. Edit the file:

```yaml
network:
    ethernets:
        eth0:
            dhcp4: yes
            optional: true
    version: 2

    wifis:
      wlan0:
        dhcp4: true
        dhcp-identifier: mac
        access-points:
          "GPA778":
            password: "GPA778Jerro2026"
```

4. Save and exit the editor: Ctrl+O, Enter, then Ctrl+X.
5. Apply the updated network configuration:
```bash
sudo netplan apply
```
6. Check the IP address assigned to the robot using:
```bash
ifconfig
```
If your IP address matches the one provided by the lab instructor...

Congratulations! Your robot is connected to the network!

#### 3.2 Remote Connection via SSH

1. On your own computer or the lab computer, open a terminal or PowerShell.
2. Connect to the robot using this SSH command:
```bash
ssh ubuntu@<your_robot_ip_address>
```  
After entering the robot's password, you should have access to the same terminal as through HDMI!


#### 3.3 Basic Commands to Try

In the previous steps, we saw several commands. Below are explanations of these and some additional useful commands.

In [ ]:
# List the contents of the current directory
ls
# Display the current working directory
pwd
# Display the command history
history
# Find previously executed commands containing the word "ros2"
history | grep "ros2"
# Display network configuration
ifconfig
# Change to a directory (here, Desktop):
cd Desktop
# Go to the parent directory
cd ..
# Display a command's manual (here, ls)
man ls
# Clear the terminal screen
clear
# List all active processes
ps -A

<a id='scp'></a>
### 3.4 File Transfer Using the SCP Protocol

SCP (Secure Copy) transfers files between the computer and the RPi.

#### General Syntax

```bash
scp [options] source destination
```

#### 3.4 Transfer from the Computer to the RPi

Example: Transfer `Bidon1.txt` from Windows to the RPi desktop:

```bash
cd .\Documents
scp .\Bidon1.txt ubuntu@10.196.129.84:/home/ubuntu/Desktop
```


#### Transfer from the RPi to the Computer

Example: Copy `Bidon2.txt` from the RPi to the Windows desktop:

```bash
cd Desktop
scp ubuntu@10.196.129.84:/home/ubuntu/Documents/Bidon2.txt .
```

The final period `.` represents the current directory.

#### Transferring Directories

To transfer an entire directory, add the `-r` (recursive) option:

```bash
scp -r ./my_directory ubuntu@10.196.129.84:/home/ubuntu/
```

**Note**: When using a hotspot, replace the address with the RPi's IP address.

### Programming on the RPi

Integrated development environments (IDEs) such as VS Code, CLion/PyCharm, and Eclipse generally support connecting to a device via SSH. For the remaining labs, we strongly recommend using VS Code with the **Remote - SSH** extension to connect the IDE to the RPi.
Screenshot from 2025-12-30 13-02-44.png

<a id='jalon3'></a>
## Milestone 3: Navigating Linux and Becoming Familiar with the Terminal

After reading the previous sections, you should now be ready to write a BASH script independently.

<a id='exercice1'></a>
### Exercise #1: File-Creation Script with Defined Permissions

Create a script named `FileCreator.bash` in the `/home/ubuntu/my_scripts/` directory.

#### Steps to Follow

Run these commands one line at a time:

```bash
cd                       # Return to the home directory
mkdir my_scripts         # Create the my_scripts directory
cd my_scripts            # Change to my_scripts
touch FileCreator.bash   # Create an empty file
chmod 744 FileCreator.bash  # Permissions: read, write, and execute for the owner
nano FileCreator.bash    # Open the nano editor
```

#### Script Contents

Your script must perform the following operations:

1. Attempt to create the `/home/ubuntu/my_scripts/fichiers/` directory
2. Ask the user to enter a filename to create
3. Create the file in the `fichiers` directory
4. Set the file permissions to **666** (read and write for everyone)
5. Display the directory contents using `ls -l`
6. Finish with `exit 0`

#### Example Script

```bash
#!/bin/bash

# Create the fichiers directory (if it does not already exist)
mkdir -p /home/ubuntu/my_scripts/fichiers/

# Ask for the filename
echo "Enter the name of the file to create:"
read filename

# Create the file
touch /home/ubuntu/my_scripts/fichiers/$filename

# Change the permissions
chmod 666 /home/ubuntu/my_scripts/fichiers/$filename

# Display the directory contents
echo "Contents of the fichiers directory:"
ls -l /home/ubuntu/my_scripts/fichiers/

exit 0
```

#### Test the Script

1. Save the file in nano: `Ctrl+X` → `Y` → `ENTER`
2. Run the script:

```bash
./FileCreator.bash
```

3. Once verified, transfer it to the lab computer using SCP

### Example Interactive Script

Let's create a simple script that asks for your name:
```bash
# Example of a simple BASH script

echo "What is your first name?"
read prenom
echo "What is your last name?"
read nom
echo "Thank you; please wait a moment"
sleep 2
echo "Your full name is $prenom $nom"
echo
echo "Goodbye"
```

<a id='jalon4'></a>
## Milestone 4: First Contact with ROS

For this final milestone, we will create a first **ROS package** and establish **inter-process communication**.

ROS 2 is already preinstalled on the RPis. Now you need to:
1. Create a ROS workspace
2. Create a package
3. Program two nodes in Python:
   - **Publisher** (*talker*): Publishes data
   - **Subscriber** (*listener*): Receives and displays data

<a id='exercice2'></a>
### Exercise #2: Creating the "bavardage" ROS Package
This exercise is adapted from the ROS 2 Humble tutorial: https://docs.ros.org/en/humble/Tutorials/Beginner-Client-Libraries/Writing-A-Simple-Py-Publisher-And-Subscriber.html

#### Step 1: Initialize the Workspace

Create a ROS 2 workspace named `ros2_ws`:

```bash
# Create the directory structure
mkdir -p /home/ubuntu/ros2_ws/src

```

#### Step 2: First Build

Build the empty workspace:
```bash
## Add ROS 2 commands to your environment
source /opt/ros/humble/setup.bash

## Go to your workspace and build it
cd /home/ubuntu/ros2_ws/
colcon build
```

#### Step 3: Initialize Your Environment

**Important**: After the **first** build, run these commands to automatically source the ROS packages whenever you open a terminal. Do this only once:
```bash
echo "source /home/ubuntu/ros2_ws/install/setup.bash" >> ~/.bashrc
echo "source /opt/ros/humble/setup.bash" >> ~/.bashrc
echo "export ROS_DOMAIN_ID=<robot_number>" >> ~/.bashrc
source ~/.bashrc
```

ROS 2 uses a Data Distribution Service (DDS) to communicate between
nodes. For more information, see this article explaining the architectural choice: https://design.ros2.org/articles/ros_on_dds.html.

We will keep the explanation simple for this course.
DDS allows a program to publish to or subscribe to a `topic` to write or read
messages. This technology uses UDP on the local network.
DDS automatically distributes discovery information so ROS 2 nodes can find one another
without manual configuration.

This is useful for communication among ROS nodes on the same network, but it can
cause interference when multiple independent systems share that network, as do the 12 robots in
this course. By default, every robot could receive messages from every other robot,
which is undesirable.

To solve this problem, systems can be isolated using a domain ID. Set the
`ROS_DOMAIN_ID` environment variable whenever you open a new terminal.
```bash
echo "export ROS_DOMAIN_ID=your_robot_number" >> ~/.bashrc
source ~/.bashrc
```

#### Step 4: Create the Package

Create the `bavardage` package:
```bash
cd /home/ubuntu/ros2_ws/src
ros2 pkg create --build-type ament_python bavardage
```

This creates a new `bavardage` directory containing two automatically generated files:
- `setup.py`: Build configuration
- `package.xml`: Package metadata

#### Step 5: Create the Python Files

Create the directory structure for the Python nodes:
```bash
# Create the Python files
cd /home/ubuntu/ros2_ws/src/bavardage/bavardage/
touch parleur.py auditeur.py
```

#### Step 6: Implement the "parleur" Node

Contents of `parleur.py`:

```python
import rclpy
from rclpy.node import Node

from std_msgs.msg import String


class Parleur(Node):

    def __init__(self):
        super().__init__('parleur')
        self.publisher_ = self.create_publisher(String, 'bavardage', 10)
        timer_period = 0.5  # seconds
        self.timer = self.create_timer(timer_period, self.timer_callback)
        self.i = 0

    def timer_callback(self):
        msg = String()
        msg.data = 'Hello World: %d' % self.i
        self.publisher_.publish(msg)
        self.get_logger().info('Publishing: "%s"' % msg.data)
        self.i += 1


def main(args=None):
    rclpy.init(args=args)

    parleur = Parleur()

    rclpy.spin(parleur)

    # Destroy the node explicitly
    # (optional - otherwise it will be done automatically
    # when the garbage collector destroys the node object)
    parleur.destroy_node()
    rclpy.shutdown()


if __name__ == '__main__':
    main()

```

#### Explanation of the parleur Code

The first lines of code import `rclpy` so that its `Node` class can be used.

```python
import rclpy
from rclpy.node import Node
```

The next statement imports the built-in string message type used by the node to structure the data it publishes on the topic.

```python
from std_msgs.msg import String
```

These lines represent the node's dependencies. Remember to add dependencies to `package.xml` in the next section.

Next, the `Parleur` class is created as a subclass of `Node`.

```python
class Parleur(Node):
```

Next, the class constructor is defined. `super().__init__` calls the `Node` class constructor and assigns the node its name, in this case `parleur`.

`create_publisher` declares that the node publishes `String` messages (imported from `std_msgs.msg`) on a topic named `bavardage`, with a queue size of 10. Queue size is a required QoS (Quality of Service) setting that limits how many messages can wait if a subscriber does not receive them quickly enough.

Next, a timer is created with a callback that runs every 0.5 seconds. `self.i` is a counter used in the callback.

```python
def __init__(self):
    super().__init__('parleur')
    self.publisher_ = self.create_publisher(String, 'bavardage', 10)
    timer_period = 0.5  # seconds
    self.timer = self.create_timer(timer_period, self.timer_callback)
    self.i = 0
```

`timer_callback` creates a message containing the counter value and prints a log message to the console using `get_logger().info`.

```python
def timer_callback(self):
    msg = String()
    msg.data = 'Hello World: %d' % self.i
    self.publisher_.publish(msg)
    self.get_logger().info('Publishing: "%s"' % msg.data)
    self.i += 1
```

Finally, the main function is defined.

```python
def main(args=None):
    rclpy.init(args=args)

    parleur = Parleur()

    rclpy.spin(parleur)

    # Destroy the node explicitly
    # (optional - otherwise it will be done automatically
    # when the garbage collector destroys the node object)
    parleur.destroy_node()
    rclpy.shutdown()
```

First, the `rclpy` library is initialized, then the node is created, and `spin` keeps the node running so its callbacks can be called.

#### Step 7: Implement the "auditeur" Node

Contents of `auditeur.py`:

```python
import rclpy
from rclpy.node import Node

from std_msgs.msg import String


class Auditeur(Node):

    def __init__(self):
        super().__init__('auditeur')
        self.subscription = self.create_subscription(
            String,
            'bavardage',
            self.listener_callback,
            10)
        self.subscription  # prevent unused variable warning

    def listener_callback(self, msg):
        self.get_logger().info('I heard: "%s"' % msg.data)


def main(args=None):
    rclpy.init(args=args)

    auditeur = Auditeur()

    rclpy.spin(auditeur)

    # Destroy the node explicitly
    # (optional - otherwise it will be done automatically
    # when the garbage collector destroys the node object)
    auditeur.destroy_node()
    rclpy.shutdown()


if __name__ == '__main__':
    main()
```

#### Explanation of the auditeur Code

The subscriber node's code is almost identical to the publisher's. Its constructor creates a subscription with the same arguments as the publisher. Remember from the topic tutorial that the topic name and message type used by the publisher and subscriber must match for them to communicate.

```python
self.subscription = self.create_subscription(
    String,
    'bavardage',
    self.listener_callback,
    10)
```

The subscriber's constructor and callback do not define a timer because one is not needed. Its callback is called whenever a message arrives.

The callback simply prints an informational message to the console along with the data it received. Recall that the publisher sets `msg.data = 'Hello World: %d' % self.i`

```python
def listener_callback(self, msg):
    self.get_logger().info('I heard: "%s"' % msg.data)
```

The `main` function is almost identical, replacing creation and spinning of the publisher with those of the subscriber.

```python
auditeur = Auditeur()

rclpy.spin(auditeur)
```

Because this node has the same dependencies as the publisher, nothing new needs to be added to `package.xml`. The `setup.cfg` file can also remain unchanged.

#### Step 8: Edit the Configuration Files
Edit `package.xml` and add:
```
<exec_depend>rclpy</exec_depend>
<exec_depend>std_msgs</exec_depend>
```
Also add an email address
```
<maintainer email="todo@todo.todo">
```

Edit `/home/ubuntu/ros2_ws/src/bavardage/setup.py` and add the following (pay attention to indentation):

```python
    entry_points={
        'console_scripts': [
            'parleur = bavardage.parleur:main',
            'auditeur = bavardage.auditeur:main',
        ],
```


#### Step 9: Build the Package

Return to the workspace directory and build the package:
```bash
cd /home/ubuntu/ros2_ws
colcon build
```


#### Step 10: Run the Nodes

You will need **two terminals** to run your package:

**Terminal 2** - Run the parleur node:
```bash
ros2 run bavardage parleur
```

**Terminal 3** - Run the auditeur node:
```bash
ros2 run bavardage auditeur
```

#### Expected Result

- **Publisher terminal**: Displays "Hello World!" messages with timestamps
- **Subscriber terminal**: Displays received messages with the node identifier

You have successfully established your first **inter-process communication** using ROS!

<a id='demo'></a>
## Demonstration to the Lab Instructor

At the end of this lab, demonstrate the following to the lab instructor:

### ✓ Demonstration 1
Show that the **Exercise #1** script (`FileCreator.bash`) works and that you have transferred the file to the lab's Windows computer.

### ✓ Demonstration 2
Show that your **"bavardage" package** works and that inter-process communication is established between the parleur and auditeur nodes.

<a id='questions'></a>
## Additional Questions


### Question 1

Run the command:

```bash
ros2 topic list
```

**What do you think the output of this command represents?**




### Question 2

Run the command:

```bash
ros2 topic echo /bavardage
```

**What output do you get, and how do you interpret it?**

*Your answer:*



### Question 3

Run the command:

```bash
ros2 topic hz /bavardage
```

**What output do you get, and how do you interpret it?**

*Your answer:*



### Question 4

Run the command:

```bash
ros2 topic info --verbose /bavardage
```

**What output do you get, and how do you interpret it?**

*Your answer:*



### Question 5

Run the command:

```bash
ros2 topic type /bavardage
```

**What output do you get, and how do you interpret it?**

*Your answer:*



### Question 6

Run the command:

```bash
rosrun rqt_graph rqt_graph
```

**In your own words, can you explain what information this command will display in the new window?**

*Your answer:*



---

## Summary of Skills Acquired

By the end of this lab, you have learned to:

✅ Understand the Raspberry Pi 4 hardware architecture

✅ Connect to the RPi via VNC and SSH, and transfer files using SCP

✅ Navigate Linux using the BASH terminal

✅ Create BASH scripts with permissions

✅ Create a ROS 2 workspace

✅ Develop a ROS package using pub/sub communication

✅ Use ROS tools to inspect topics

**Congratulations! You are now ready for the next labs!** 🎉